In [1]:
##################
### Keagan Moo ###
### 2022/07/XX ###
### Linear Mod ###
##################

# This script runs the pseudonulk age-association analysis with DESeq2 for use in comparison to the single nucleus-level GLMM model

options(stringsAsFactors = F)
suppressPackageStartupMessages(library(edgeR))
suppressPackageStartupMessages(library(glue))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(glue))
suppressPackageStartupMessages(library(zoo))
suppressPackageStartupMessages(library(forecast))
suppressPackageStartupMessages(library(parallel))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(DESeq2))
suppressPackageStartupMessages(library("Matrix.utils"))
suppressPackageStartupMessages(library("SingleCellExperiment"))



Warning message:
“package ‘Matrix.utils’ was built under R version 4.4.3”


In [2]:
feature <- "Type_2x"
### LOAD AND SHAPE DATA ###

# Load and organize metadata so that it matches the ARIMA set
# KM # Loads the clustering data, presumably form DESeq non sex specific, in
# KM # a format unfamiliar to me. It appears to be the individual nuclei in rows
# KM # with QA and annotations as features. Some map to the same donor, QA
# KM # includes mitochondrial content.
barcode_md <- file.path(
  "/nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/freeze/qc_fusion.bgstringent__k_21",
  paste0("cluster_info_qc.tsv")
) %>% 
  fread(data.table = F)

covariate_md <- file.path(
  "/nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/freeze",
  paste0("sample_level_covariates_atac.tsv")
) %>% 
  fread(data.table = F)

covSNG <- data.frame(SNG = covariate_md$SNG.1ST, cluster = substr(covariate_md$coarse_cluster_name, 8, nchar(covariate_md$coarse_cluster_name)),
                     age = covariate_md$age, sex = covariate_md$sex, bmi = covariate_md$bmi, tss_enrichment_sampleCluster = covariate_md$tss_enrichment)

barcode_md$ID <- paste0(barcode_md$SNG.1ST, barcode_md$coarse_cluster_name)
covSNG$ID <- paste0(covSNG$SNG, covSNG$cluster)
covSNG <- covSNG[,!colnames(covSNG) %in% c("SNG", "cluster")]

barcode_md <- merge(barcode_md, covSNG, by = "ID")
barcode_md <- barcode_md[,!colnames(barcode_md) %in% c("ID")]

# KM # Does what it says on the tin. Groups the cellular data by sample
# KM # generates (mutates) a new column for median tss, distinct "fins" values
# KM # and assigns both forms to a new variable. The broad purpose seems to 
# KM # be to make this data conform to a standard "ucar" that I haven't used
metadata.factors <- barcode_md %>%
  group_by(sample) %>%
  mutate(tss_enrich_median = median(tss_enrichment, na.rm = T),
         n_nuclei = n()) %>%
  # cohort:fins gets all columns in this range, including sex, age, etc.
  # we can just get the fins value from the prior dataset but we need to make
  # sure to keep all the columns in between here.
  dplyr::select(sample, batch, cohort, age, sex, bmi, tss_enrich_median, n_nuclei) %>%
  distinct() %>%
  
  # fix wrong sex assignments
  mutate(sex = case_when(sample %in% c("SA02842--564", "SA02842--625") ~ "M",
                         T ~ sex)) %>%
  
  # Keep only FUSION samples
  dplyr::filter(cohort %in% "FUSION") %>%
  
  # Format for model matrices
  mutate(sex = factor(sex, levels = c("M", "F"))) %>%
  mutate(n_nuclei_l10 = log10(n_nuclei)) %>%
  dplyr::filter(!is.na(age)) %>%
  # mutate(age.group = cut(age, seq(20, 80, 5), right = F, ordered_result = T)) %>%
  mutate(age.group = case_when(age <= 40 ~ "young",
                               age > 40 & age <= 64 ~ "midage",
                               age >= 65 ~ "old")) %>%
  mutate(age.group = factor(age.group, levels = c("young", "midage", "old"), ordered = T)) %>%
  mutate(age.group = as.numeric(age.group)) %>%  # yields more readable model matrices
  arrange(age) %>%
  as.data.frame() %>%
  
  # conform to Ucar script
  mutate(sampid = sample,
         batchdate = batch,
         nreps = 1)

# KM # Load in arushi's cell type specific TSS enrichment data so we can use that
TSSDat <- read.table("/nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/freeze/cluster-sample-ataqv-stats.txt", header = TRUE)
nameFrame <- data.frame(t(data.frame(strsplit(TSSDat$name, "@"))))
rownames(nameFrame) <- NULL
colnames(nameFrame) <- c("Type","Cohort","sample")
TSSDat$Type <- nameFrame$Type
TSSDat$sample <- nameFrame$sample
TSSDat <- TSSDat[TSSDat$Type == glue("{feature}"),]
TSSDat <- TSSDat %>% dplyr::select(c("sample", "tss_enrichment"))
metadata.factors <- merge(metadata.factors, TSSDat, by = "sample")

# KM # Load in T2D sample data to include in correction
SampleData <- read.table("/nfs/turbo/umms-scjp/arushiv/projects/muscle-sn/analyses_hg38/freeze/batch-sample-info-full.tsv", sep = "\t", header = TRUE)
SampleData <- SampleData[SampleData$sample %in% metadata.factors$sample,]
SampleData <- SampleData %>% dplyr::select(c("sample", "ogtt"))

metadata.factors <- merge(metadata.factors, SampleData, by = "sample")
rownames(metadata.factors) <- metadata.factors$sample

metadata.factors$batch <- as.factor(metadata.factors$batch)



In [3]:
small <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/{feature}_FUSIONsnATAC_sce.rds"))
smallF <- small[,small$sex == "F"]
smallM <- small[,small$sex == "M"]
keepRows <- rowSums(counts(small) >= 1) >= ncol(counts(small))*0.1
small <- small[keepRows,]
keepRowsF <- rowSums(counts(smallF) >= 1) >= ncol(counts(smallF))*0.1
smallF <- smallF[keepRowsF,]
keepRowsM <- rowSums(counts(smallM) >= 1) >= ncol(counts(smallM))*0.1
smallM <- smallM[keepRowsM,]

groups <- colData(small)[, c("sample")]
peaks <- aggregate.Matrix(t(counts(small)), 
                       groupings = groups, fun = "sum")
groupsF <- colData(smallF)[, c("sample")]
peaksF <- aggregate.Matrix(t(counts(smallF)), 
                       groupings = groupsF, fun = "sum")
groupsM <- colData(smallM)[, c("sample")]
peaksM <- aggregate.Matrix(t(counts(smallM)), 
                       groupings = groupsM, fun = "sum")

In [4]:
peaks <- t(peaks)
peaksF <- t(peaksF)
peaksM <- t(peaksM)

In [7]:
# Load and append peak data in a form matching the ARIMA stuff
peaks_dirS <- file.path("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/Files/ARIMAke/",
                       "arushivData20220415/fusion_150_ccc")

peaksS <- file.path(peaks_dirS, glue("atac.{feature}.ccc.tsv")) %>%
  fread(data.table = F) %>%
  tibble::column_to_rownames("feature") %>%
  data.matrix()

peaksS <- peaksS[, colnames(peaksS) != "V1"]

Warning message in fread(., data.table = F):
“Detected 286 column names but the data has 287 columns (i.e. invalid file). Added 1 extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.”


In [6]:
fPeaks <- peaksF
mPeaks <- peaksM

In [10]:
dim(peaks)
head(peaks)

[1] 41758   284

  [[ suppressing 284 column names ‘12001--510’, ‘12002--551’, ‘12003--530’ ... ]]



6 x 284 sparse Matrix of class "dgCMatrix"
                                                                              
chr1:1000015:1000316      22 11  30  44 30  23  66  35 .  86 48  50  65  59 22
chr1:100037924:100038225  95 31 117 126 73  65 101 138 . 173 39  78 101  81 55
chr1:100132863:100133164 158 61 197 160 88 109 159 161 . 290 82 116 134 118 83
chr1:100178145:100178446  34 14  33  20 35  22  35  35 .  51 20  49  36  19 10
chr1:1001840:1002141      37 26  71  65 40  41  71  60 . 101 30  48  51  42 24
chr1:100187982:100188283  19  4  28  17 21  18  28  12 .  29 24  18  33   5  7
                                                                              
chr1:1000015:1000316     .  70  51 14  35 17  65  72 17 38  44  31  43 106  57
chr1:100037924:100038225 1  72 106 58  76 41  71 150 66 49  92  97  85 155 102
chr1:100132863:100133164 3 126 145 78 118 75 111 184 67 99 124 153 152 263 134
chr1:100178145:100178446 .  19  30 16  16 21  17  35  9 26  33  31  39  45  15
chr1:1001

In [4]:
# Put everything in the same order and remove samples with missing age
peaks <- peaks[,colnames(peaks) %in% rownames(metadata.factors)]
metadata.factors <- metadata.factors[metadata.factors$sample %in% colnames(peaks),]
peaks <- peaks[,rownames(metadata.factors)]

if (!all(rownames(metadata.factors) == colnames(peaks))) {
  stop("Something wrong (1)")
}

libsizes <- colSums(peaks)

if (all(names(libsizes) == metadata.factors$sample)) {
  message("Adding library sizes to meta data")
  metadata.factors$libsize <- log2(unname(libsizes))
} else {
  stop("Something wrong (2)")
}

Adding library sizes to meta data



In [7]:
# Peaks are sparse, such that in some cell types a sample has 0 counts in every peak... apparently, which causes DESeq2 to fail. This is the solution from online
peaks <- peaks + 1
fPeaks <- fPeaks + 1
mPeaks <- mPeaks + 1

In [8]:
### Begin sex specific breakout ###

sampleDepths <- colSums(peaks)
metadata.factors$sDepth <- sampleDepths

#fPeaks <- peaks[,colnames(peaks) %in% metadata.factors$sample[metadata.factors$sex == "F"]]
#mPeaks <- peaks[,colnames(peaks) %in% metadata.factors$sample[metadata.factors$sex == "M"]]

libsizes <- colSums(peaks)
fLibsizes <- colSums(fPeaks)
mLibsizes <- colSums(mPeaks)

metadata.factors$ogtt <- as.factor(metadata.factors$ogtt)
head(metadata.factors)

,sample,batch,cohort,age,sex,bmi,tss_enrich_median,n_nuclei,n_nuclei_l10,age.group,sampid,batchdate,nreps,tss_enrichment,ogtt,sDepth
,<chr>,<fct>,<chr>,<dbl>,<fct>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<chr>,<int>,<dbl>,<dbl>,<fct>,<dbl>
12001--510,12001--510,510,FUSION,66,M,30.37742,5.442277,1963,3.292920,3,12001--510,510,1,5.637468,IGT,4148091
12002--551,12002--551,551,FUSION,61,M,25.10174,4.867827,1095,3.039414,2,12002--551,551,1,5.468351,NGT,3456178
12003--530,12003--530,530,FUSION,58,M,25.51226,6.965487,2632,3.420286,2,12003--530,530,1,6.924750,NGT,5438155
12004--530,12004--530,530,FUSION,64,M,23.03005,5.393279,2497,3.397419,2,12004--530,530,1,6.156195,NGT,4749899
12006--530,12006--530,530,FUSION,56,F,22.79218,6.327894,1870,3.271842,2,12006--530,530,1,6.536359,NGT,705669
12007--647,12007--647,647,FUSION,64,F,21.35780,4.670505,1604,3.205204,2,12007--647,647,1,4.787282,IGT,2155361


In [11]:
head(sampleDepths)
head(metadata.factors)

12001--510 12002--551 12003--530 12004--530 12006--530 12007--647 
   3696829    1595444    4298618    3559485    2305408    2117984

,sample,batch,cohort,age,sex,bmi,tss_enrich_median,n_nuclei,n_nuclei_l10,age.group,sampid,batchdate,nreps,tss_enrichment,ogtt,libsize,sDepth
,<chr>,<fct>,<chr>,<dbl>,<fct>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<chr>,<int>,<dbl>,<dbl>,<fct>,<dbl>,<dbl>
12001--510,12001--510,510,FUSION,66,M,30.37742,5.442277,1963,3.292920,3,12001--510,510,1,6.373854,IGT,21.75820,3696829
12002--551,12002--551,551,FUSION,61,M,25.10174,4.867827,1095,3.039414,2,12002--551,551,1,6.080952,NGT,20.46333,1595444
12003--530,12003--530,530,FUSION,58,M,25.51226,6.965487,2632,3.420286,2,12003--530,530,1,7.855721,NGT,21.98429,4298618
12004--530,12004--530,530,FUSION,64,M,23.03005,5.393279,2497,3.397419,2,12004--530,530,1,7.413127,NGT,21.70123,3559485
12006--530,12006--530,530,FUSION,56,F,22.79218,6.327894,1870,3.271842,2,12006--530,530,1,7.634585,NGT,21.03970,2305408
12007--647,12007--647,647,FUSION,64,F,21.35780,4.670505,1604,3.205204,2,12007--647,647,1,5.498161,IGT,20.90847,2117984


In [9]:
fMeta <- metadata.factors[metadata.factors$sex == "F",]
mMeta <- metadata.factors[metadata.factors$sex == "M",]

ddsTotal <- DESeqDataSetFromMatrix(countData = peaks,
                              colData = metadata.factors,
                              design = ~ sex + bmi + batch + n_nuclei_l10 + tss_enrichment + ogtt + age)
ddsFemale <- DESeqDataSetFromMatrix(countData = fPeaks,
                              colData = fMeta,
                              design = ~ bmi + batch + n_nuclei_l10 + tss_enrichment + ogtt + age)
ddsMale <- DESeqDataSetFromMatrix(countData = mPeaks,
                              colData = mMeta,
                              design = ~ bmi + batch + n_nuclei_l10 + tss_enrichment + ogtt + age)

# COMMENTED OUT ONLY FOR GLMMNB PEAKS TEST AS PEAKS ARE ALREADY FILTERED
#checkSize <- round(nrow(metadata.factors)*0.10)
#keep <- rowSums(counts(ddsTotal) >= 10) >= checkSize
#ddsTotal <- ddsTotal[keep,]

#checkSizeF <- round(nrow(fMeta)*0.10)
#keepF <- rowSums(counts(ddsFemale) >= 10) >= checkSizeF
#ddsFemale <- ddsFemale[keepF,]

#checkSizeM <- round(nrow(mMeta)*0.10)
#keepM <- rowSums(counts(ddsMale) >= 10) >= checkSizeM
#ddsMale <- ddsMale[keepM,]

ddsTotal <- DESeq(ddsTotal)
resTotal <- results(ddsTotal)
resTotalOrdered <- resTotal[order(resTotal$pvalue),]
resTotalShrunk <- lfcShrink(ddsTotal, coef="age", type="ashr")
resTotalShrunkOrdered <- resTotalShrunk[order(resTotalShrunk$pvalue),]

ddsFemale <- DESeq(ddsFemale)
resFemale <- results(ddsFemale)
resFemaleOrdered <- resFemale[order(resFemale$pvalue),]
resFemaleShrunk <- lfcShrink(ddsFemale, coef="age", type="ashr")
resFemaleShrunkOrdered <- resFemaleShrunk[order(resFemaleShrunk$pvalue),]

ddsMale <- DESeq(ddsMale)
resMale <- results(ddsMale)
resMaleOrdered <- resMale[order(resMale$pvalue),]
resMaleShrunk <- lfcShrink(ddsMale, coef="age", type="ashr")
resMaleShrunkOrdered <- resMaleShrunk[order(resMaleShrunk$pvalue),]

converting counts to integer mode

  the design formula contains one or more numeric variables with integer values,
  specifying a model with increasing fold change for higher values.
  did you mean for this to be a factor? if so, first convert
  this variable to a factor using the factor() function

  the design formula contains one or more numeric variables that have mean or
  standard deviation larger than 5 (an arbitrary threshold to trigger this message).
  Including numeric variables with large mean can induce collinearity with the intercept.
  Users should center and scale numeric variables in the design to improve GLM convergence.

converting counts to integer mode

  the design formula contains one or more numeric variables with integer values,
  specifying a model with increasing fold change for higher values.
  did you mean for this to be a factor? if so, first convert
  this variable to a factor using the factor() function

  the design formula contains one or more numeric 

In [8]:
post_counts <- data.frame(colSums(counts(ddsFemale)))
colnames(post_counts) <- "pDepth"
metadata.FactorsPlot <- merge(fMeta, post_counts, by = "row.names")
post_counts <- data.frame(colSums(counts(ddsMale)))
colnames(post_counts) <- "pDepth"
Metadata.factorsPlot <- merge(mMeta, post_counts, by = "row.names")
metadata.FactorsPlot$lossP <- (1-(metadata.FactorsPlot$pDepth/metadata.FactorsPlot$sDepth))
Metadata.factorsPlot$lossP <- (1-(Metadata.factorsPlot$pDepth/Metadata.factorsPlot$sDepth))

In [9]:
cor.test(metadata.FactorsPlot$age, metadata.FactorsPlot$pDepth, method = "spearman")
cor.test(Metadata.factorsPlot$age, Metadata.factorsPlot$pDepth, method = "spearman")

Warning message in cor.test.default(metadata.FactorsPlot$age, metadata.FactorsPlot$pDepth, :
“Cannot compute exact p-value with ties”



	Spearman's rank correlation rho

data:  metadata.FactorsPlot$age and metadata.FactorsPlot$pDepth
S = 295437, p-value = 0.7789
alternative hypothesis: true rho is not equal to 0
sample estimates:
        rho 
-0.02589329 


Warning message in cor.test.default(Metadata.factorsPlot$age, Metadata.factorsPlot$pDepth, :
“Cannot compute exact p-value with ties”



	Spearman's rank correlation rho

data:  Metadata.factorsPlot$age and Metadata.factorsPlot$pDepth
S = 667573, p-value = 0.2419
alternative hypothesis: true rho is not equal to 0
sample estimates:
       rho 
0.09189773 


In [15]:
pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/KMScripts/Plots/logpreCountsVAge_Type2a_F_Samp.pdf"), width=6, height=6)
    mygg <- ggplot(data=metadata.FactorsPlot, aes(x=age, y=log(sDepth))) +
        geom_point(alpha = 0.3, position = position_jitter(width = 0.01)) +
        theme_bw() + #xlim(-11,11) + ylim(-11,11) +
        xlab("Age") + ylab("% Counts lost after peak filtration") + ggtitle(glue("Age v % Total Count Loss")) 
   print(mygg)
dev.off()

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/KMScripts/Plots/logpreCountsVAge_Type2a_M_Samp.pdf"), width=6, height=6)
    mygg <- ggplot(data=Metadata.factorsPlot, aes(x=age, y=log(sDepth))) +
        geom_point(alpha = 0.3, position = position_jitter(width = 0.01)) +
        theme_bw() + #xlim(-11,11) + ylim(-11,11) +
        xlab("Age") + ylab("% Counts lost after peak filtration") + ggtitle(glue("Age v % Total Count Loss")) 
   print(mygg)
dev.off()

pdf 
  2

pdf 
  2

In [11]:
pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/KMScripts/Plots/FilterCountsLossPreVPost_Type2a_F_Samp.pdf"), width=6, height=6)
    mygg <- ggplot(data=metadata.FactorsPlot, aes(x=sDepth, y=pDepth)) +
        geom_point(alpha = 0.3) +
        theme_bw() + #xlim(-11,11) + ylim(-11,11) +
        geom_abline(intercept = 0, slope = 1, alpha = 1, col = "black") +
        xlab("Pre Filter Total Counts") + ylab("Post Filter Total Counts") 
   print(mygg)
dev.off()
pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/KMScripts/Plots/FilterCountsLossPreVPost_Type2a_M_Samp.pdf"), width=6, height=6)
    mygg <- ggplot(data=Metadata.factorsPlot, aes(x=sDepth, y=pDepth)) +
        geom_point(alpha = 0.3) +
        theme_bw() + #xlim(-11,11) + ylim(-11,11) +
        geom_abline(intercept = 0, slope = 1, alpha = 1, col = "black") +
        xlab("Pre Filter Total Counts") + ylab("Post Filter Total Counts") 
   print(mygg)
dev.off()

pdf 
  2

pdf 
  2

In [18]:
#head(resFemaleShrunkOrdered, n = 10)
resTotalShrunk$padj[is.na(resTotalShrunk$padj)] <- 1
resFemaleShrunk$padj[is.na(resFemaleShrunk$padj)] <- 1
resMaleShrunk$padj[is.na(resMaleShrunk$padj)] <- 1


nrow(resTotalShrunk[resTotalShrunk$padj <= 0.1,])

nrow(resFemaleShrunk[resFemaleShrunk$padj <= 0.1,])
nrow(resFemaleShrunk[(resFemaleShrunk$padj <= 0.1) & (resFemaleShrunk$log2FoldChange < 0),])
nrow(resFemaleShrunk[(resFemaleShrunk$padj <= 0.1) & (resFemaleShrunk$log2FoldChange > 0),])

nrow(resMaleShrunk[resMaleShrunk$padj <= 0.1,])
nrow(resMaleShrunk[(resMaleShrunk$padj <= 0.1) & (resMaleShrunk$log2FoldChange < 0),])
nrow(resMaleShrunk[(resMaleShrunk$padj <= 0.1) & (resMaleShrunk$log2FoldChange > 0),])

[1] 6184

[1] 900

[1] 591

[1] 309

[1] 1814

[1] 1062

[1] 752

In [10]:
saveRDS(list(resTotalShrunk, resFemaleShrunk, resMaleShrunk) , glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/{feature}_DESeq2_GLMMNBPeaks.rds"))
#saveRDS(list(peaksAdjusted, fPeaksAdjusted, mPeaksAdjusted), glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/{feature}_glmoffadj.rds"))

In [4]:
linearRes <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/{feature}_DESeq2.rds"))

LF <- linearRes[[2]][linearRes[[2]]$pvalue < 0.1,]
LM <- linearRes[[3]][linearRes[[3]]$pvalue < 0.1,]

LF <- LF[order(LF$pvalue),]
LM <- LM[order(LM$pvalue),]

LFO <- LF[LF$log2FoldChange > 0,]
LFC <- LF[LF$log2FoldChange < 0,]
LMO <- LM[LM$log2FoldChange > 0,]
LMC <- LM[LM$log2FoldChange < 0,]

Pattern <- c("OPEN", "CLOSE", "OPEN", "CLOSE")
sexCol <- c("female","female","male","male")
countCol <- c(length(rownames(LFO)), 
                      length(rownames(LFC)), 
                      length(rownames(LMO)), 
                      length(rownames(LMC)))
dfPlot <- data.frame(Pattern, sexCol, countCol)               
dfPlot$Combi <- factor(paste0(dfPlot$Pattern, "-", dfPlot$sexCol), levels = c("OPEN-female", "CLOSE-female", "OPEN-male", "CLOSE-male"))
dfPlot

Pattern,sexCol,countCol,Combi
<chr>,<chr>,<int>,<fct>
OPEN,female,4980,OPEN-female
CLOSE,female,4501,CLOSE-female
OPEN,male,8986,OPEN-male
CLOSE,male,8378,CLOSE-male


In [15]:
ColorCode_FemaleUP <- "#CC6677"
ColorCode_FemaleDOWN <- "#4AE4CB"
ColorCode_MaleUP <- "#882255"
ColorCode_MaleDOWN <- "#117733"

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Plots/AgeATAC/DESeq2PeakCompare_{feature}.pdf"), width=6, height=6)
mygg <- ggplot(dfPlot, aes(x = Combi, y = countCol, fill = Combi)) +
  geom_bar(position = 'dodge', stat = "identity") +
  xlab("Chromatin Behavior by Sex") + ylab("Number of Peaks") + ggtitle(glue("Significant FDR < 0.1 {feature} Peaks")) + 
  scale_fill_manual(values = c(ColorCode_FemaleUP,ColorCode_FemaleDOWN,ColorCode_MaleUP, ColorCode_MaleDOWN)) + 
  scale_y_continuous(breaks=seq(0,2000,100), limits = c(0,2000)) + theme_bw()
mygg
dev.off()

pdf 
  2

In [15]:
head(LM[LM$log2FoldChange < 0,], n = 5)
nrow(LF[LF$log2FoldChange < 0,])
nrow(LF[LF$log2FoldChange > 0,])
nrow(LM[LM$log2FoldChange < 0,])
nrow(LM[LM$log2FoldChange > 0,])

log2 fold change (MMSE): age 
Wald test p-value: age 
DataFrame with 5 rows and 5 columns
                          baseMean log2FoldChange      lfcSE      pvalue
                         <numeric>      <numeric>  <numeric>   <numeric>
chr15:41557654:41557955    3.98196    -0.00187000 0.00288491 3.45772e-05
chr6:158825983:158826284   4.01809    -0.00175369 0.00281287 3.57981e-05
chr12:75706003:75706304    6.29186    -0.00229022 0.00311051 3.82852e-05
chr9:34084559:34084860     5.64990    -0.00184600 0.00286797 3.98682e-05
chr1:16226866:16227167     5.57651    -0.00199465 0.00294899 5.18000e-05
                              padj
                         <numeric>
chr15:41557654:41557955   0.436342
chr6:158825983:158826284  0.436342
chr12:75706003:75706304   0.436342
chr9:34084559:34084860    0.436342
chr1:16226866:16227167    0.436342

[1] 1543

[1] 1713

[1] 3148

[1] 2582

In [8]:
head(peaks)

,12001--510,12002--551,12003--530,12004--530,12006--530,12007--647,12008--530,12009--499,12010--647,12011--499,⋯,32113--692,32114--499,32115--647,32117--530,32118--551,32119--692,32120--551,32121--510,P14744--530,P14762--564
chr10:100000195:100000496,13,3,17,3,26,23,3,9,0,13,⋯,11,9,22,22,17,20,19,15,9,28
chr10:100009814:100010115,187,49,255,78,285,221,200,152,0,212,⋯,229,119,215,154,228,212,273,220,126,124
chr10:100025760:100026061,5,3,6,3,6,4,6,3,0,12,⋯,9,6,7,2,12,4,10,7,7,14
chr10:100046058:100046359,64,21,66,45,95,75,54,46,0,84,⋯,66,25,90,42,68,81,70,79,35,41
chr10:100115078:100115379,4,0,3,2,6,6,1,1,0,4,⋯,6,8,2,5,11,4,5,6,1,4
chr10:100127715:100128016,5,1,17,2,15,19,5,9,0,6,⋯,21,5,13,7,16,9,6,10,8,7


In [5]:
### PLOT PEAKS OVER AGE TO DETERMINE DISTINCTION ###
# PO # use adjusted data, specifically tss_enrich_median
# Make time series object with average values for each peak at each charted age.

# Replace the sample ID in the dataframe by their age from meta.data factors
# combine rows of the same age by taking the average
# No interpolation, no fancy continuum, maybe z-scale it and then just plot a line
# It looks like this is what the ucar lab do in arima.selections
ages <- with(metadata.factors,setNames(age,sampid))
sexes <- with(metadata.factors,setNames(sex,sampid))
# Get the list of ages in order which appear in male and female samples
mAges <- ages[names(sexes[sexes=="M"])]
fAges <- ages[names(sexes[sexes=="F"])]
# Get the subset of data that fall under that category
mPeaks <- peaks[,names(sexes[sexes=="M"])]
fPeaks <- peaks[,names(sexes[sexes=="F"])]

# List out the ages that each sex has access to and their frequencies
mAgeCounts <- as.data.frame(table(mAges),stringsAsFactors = F)
fAgeCounts <- as.data.frame(table(fAges),stringsAsFactors = F)

# Set type to numeric? I'm not sure if it's necessary
fPeaks <- t(data.frame(fPeaks))
colnames(fPeaks) <- gsub(":", ".", colnames(fPeaks))
mPeaks <- t(data.frame(mPeaks))
colnames(mPeaks) <- gsub(":", ".", colnames(mPeaks))
fAgesUnique <- as.numeric(fAgeCounts$fAges)
mAgesUnique <- as.numeric(mAgeCounts$mAges)


In [31]:
mAgesUnique
head(mPeaks)
length(mAges)
print("chr2.173399263.173399564" %in% colnames(mPeaks))
(mPeaks[,'chr2.173399263.173399564'])

[1] 25 41 42 43 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66
[26] 67 68 69 70 71 72 73 74 77 79

,chr10.100000195.100000496,chr10.100009814.100010115,chr10.100025760.100026061,chr10.100046058.100046359,chr10.100115078.100115379,chr10.100127715.100128016,chr10.100130818.100131119,chr10.100146903.100147204,chr10.100166760.100167061,chr10.100173107.100173408,⋯,chrX.97295927.97296228,chrX.97367171.97367472,chrX.9786167.9786468,chrX.9874022.9874323,chrX.9885189.9885490,chrX.9899693.9899994,chrX.990099.990400,chrX.99705507.99705808,chrX.9995839.9996140,chrX.9997502.9997803
X12001..510,13,187,5,64,4,5,37,8,5,2,⋯,6,2,13,12,15,4,8,2,65,15
X12002..551,3,49,3,21,0,1,13,2,1,2,⋯,4,1,4,4,4,4,9,1,18,2
X12003..530,17,255,6,66,3,17,63,23,4,7,⋯,8,7,8,13,17,5,21,1,78,6
X12004..530,3,78,3,45,2,2,13,9,2,1,⋯,0,1,4,8,7,3,0,8,32,2
X12008..530,3,200,6,54,1,5,19,6,2,1,⋯,3,3,6,11,14,1,41,6,61,15
X12009..499,9,152,3,46,1,9,53,20,0,2,⋯,8,1,7,13,10,6,60,8,50,10


[1] 164

[1] TRUE


X12001..510 X12002..551 X12003..530 X12004..530 X12008..530 X12009..499 
         47          10          13           3          20          15 
X12011..499 X12016..551 X12017..625 X12020..564 X12022..510 X12025..530 
         18          10           7           9          14          53 
X12029..510 X12030..530 X12033..551 X12036..678 X12037..551 X12038..647 
         43          22           8           1          33           6 
X12039..625 X12041..564 X12042..510 X12044..499 X12052..499 X12053..567 
         21          18           1          43           7          12 
X12054..647 X12057..551 X12058..564 X12060..551 X12063..510 X12069..530 
         51          13          11          22           6          16 
X12070..625 X12071..564 X12072..530 X12073..692 X12074..499 X12075..510 
         11           5           7           7           3          22 
X12077..625 X12078..692 X12079..567 X12080..564 X12081..564 X12082..551 
         11          11          18           8           7          20 
X12083..678 X12084..530 X12085..551 X12086..499 X12087..647 X12088..530 
         14          10          14         124           7          37 
X12089..567 X12090..551 X12091..678 X12092..510 X12093..510 X12094..499 
         10           7          18          41          20           1 
X12095..499 X12096..551 X12097..567 X12098..551 X12099..499 X12101..551 
          5           3           6          14           2           4 
X12102..530 X12103..551 X12104..510 X12131..499 X12132..647 X12133..692 
         87          33           9          10           2           3 
X12134..564 X12135..678 X12136..499 X12137..530 X12138..499 X12139..564 
         15          20          14           0           2           3 
X12140..647 X12141..530 X12142..567 X12143..510 X12144..530 X12145..510 
         39          52          50           5           1          11 
X12146..567 X12147..510 X12148..551 X12149..499 X12150..647 X12151..510 
          5           7           8          28           1          11 
X12155..625 X12159..564 X12162..530 X12163..499 X12164..510 X12165..499 
          7           3          15           6           3          26 
X12170..510 X12173..564 X12174..692 X12179..551 X22001..567 X22003..647 
         10          11           2           1          10          10 
X22006..551 X22008..499 X22009..692 X22016..625 X22021..510 X22027..564 
          9           2          25          14           7          17 
X22031..692 X22032..647 X22034..625 X22035..551 X22039..692 X32001..499 
          4           9           6           9           3          14 
X32002..510 X32003..530 X32004..625 X32008..564 X32009..647 X32015..551 
          2           3           2           6          26          13 
X32018..692 X32027..567 X32028..499 X32031..530 X32032..551 X32034..530 
         14           7          14          16          11          78 
X32038..564 X32039..530 X32040..499 X32041..551 X32042..678 X32043..625 
          0          14          12          23          76          16 
X32044..551 X32045..564 X32047..678 X32050..564 X32058..625 X32059..567 
          6          15           8           7          14          42 
X32060..567 X32061..530 X32064..551 X32066..510 X32068..567 X32073..499 
          6          48           4          12          26          89 
X32076..647 X32077..551 X32079..551 X32083..510 X32085..510 X32088..499 
          2          16          44          28          48          20 
X32089..510 X32090..564 X32093..567 X32096..551 X32097..564 X32098..530 
          8          19          70          24           7           4 
X32099..530 X32100..510 X32101..625 X32102..551 X32104..499 X32105..510 
         52          18          10           9          14           3 
X32106..564 X32107..530 X32108..567 X32115..647 X32117..530 X32121..510 
         10          30          27          30          10          45 
P14744..530 P14762..564 
         15          10

In [6]:
coords <- "chr3.48717169.48717470"

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Plots/AgeATAC/PeakLook/{coords}_Male_{feature}.pdf"), width=6, height=6)
mygg <- ggplot(data=mPeaks, aes(x=mAges, y=chr3.48717169.48717470)) +
  #geom_errorbar(aes(ymin = log2FC_CL, ymax = log2FC_CU)) + 
  geom_point() + theme_bw() +
  xlab("Sample Age") + ylab("Gene Counts")+ ggtitle(coords)
mygg
dev.off()

pdf 
  2

In [32]:
head(T1ResF)

log2 fold change (MMSE): age 
Wald test p-value: age 
DataFrame with 6 rows and 5 columns
                           baseMean log2FoldChange      lfcSE    pvalue
                          <numeric>      <numeric>  <numeric> <numeric>
chr10:100000195:100000496  13.05831   -2.45021e-04 0.00191937  0.652929
chr10:100009814:100010115 145.36377    3.28440e-04 0.00130728  0.678904
chr10:100025760:100026061   5.13237   -4.69593e-04 0.00227907  0.354666
chr10:100046058:100046359  45.92772   -2.34856e-04 0.00149707  0.743179
chr10:100115078:100115379   4.29999   -7.86427e-07 0.00212225  0.998607
chr10:100127715:100128016   9.60845   -8.66614e-04 0.00276081  0.125808
                               padj
                          <numeric>
chr10:100000195:100000496  0.923246
chr10:100009814:100010115  0.930743
chr10:100025760:100026061  1.000000
chr10:100046058:100046359  0.948035
chr10:100115078:100115379  1.000000
chr10:100127715:100128016  0.625254

In [21]:
T1ResF <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/Type_1_DESeq2.rds"))[[2]]
T1ResM <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/Type_1_DESeq2.rds"))[[3]]

T2aResF <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/Type_2a_DESeq2.rds"))[[2]]
T2aResM <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/Type_2a_DESeq2.rds"))[[3]]

T2xResF <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/Type_2x_DESeq2.rds"))[[2]]
T2xResM <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/Type_2x_DESeq2.rds"))[[3]]

T1ResFCBed <- t(data.frame(strsplit(rownames(T1ResF[(T1ResF$log2FoldChange < 0) & (T1ResF$padj < 0.1),]), ":")))
rownames(T1ResFCBed) <- NULL
nrow(T1ResFCBed)
write.table(T1ResFCBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_1_FC.bed", sep = "\t", row.names = FALSE)
T1ResFOBed <- t(data.frame(strsplit(rownames(T1ResF[(T1ResF$log2FoldChange > 0) & (T1ResF$padj < 0.1),]), ":")))
rownames(T1ResFOBed) <- NULL
nrow(T1ResFOBed)
write.table(T1ResFOBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_1_FO.bed", sep = "\t", row.names = FALSE)

T1ResMCBed <- t(data.frame(strsplit(rownames(T1ResM[(T1ResM$log2FoldChange < 0) & (T1ResM$padj < 0.1),]), ":")))
rownames(T1ResMCBed) <- NULL
nrow(T1ResMCBed)
write.table(T1ResMCBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_1_MC.bed", sep = "\t", row.names = FALSE)
T1ResMOBed <- t(data.frame(strsplit(rownames(T1ResM[(T1ResM$log2FoldChange > 0) & (T1ResM$padj < 0.1),]), ":")))
rownames(T1ResMOBed) <- NULL
nrow(T1ResMOBed)
write.table(T1ResMOBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_1_MO.bed", sep = "\t", row.names = FALSE)


T2aResFCBed <- t(data.frame(strsplit(rownames(T2aResF[(T2aResF$log2FoldChange < 0) & (T2aResF$padj < 0.1),]), ":")))
rownames(T2aResFCBed) <- NULL
nrow(T2aResFCBed)
write.table(T2aResFCBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_2a_FC.bed", sep = "\t", row.names = FALSE)
T2aResFOBed <- t(data.frame(strsplit(rownames(T2aResF[(T2aResF$log2FoldChange > 0) & (T2aResF$padj < 0.1),]), ":")))
rownames(T2aResFOBed) <- NULL
nrow(T2aResFOBed)
write.table(T2aResFOBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_2a_FO.bed", sep = "\t", row.names = FALSE)

T2aResMCBed <- t(data.frame(strsplit(rownames(T2aResM[(T2aResM$log2FoldChange < 0) & (T2aResM$padj < 0.1),]), ":")))
rownames(T2aResMCBed) <- NULL
nrow(T2aResMCBed)
write.table(T2aResMCBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_2a_MC.bed", sep = "\t", row.names = FALSE)
T2aResMOBed <- t(data.frame(strsplit(rownames(T2aResM[(T2aResM$log2FoldChange > 0) & (T2aResM$padj < 0.1),]), ":")))
rownames(T2aResMOBed) <- NULL
nrow(T2aResMOBed)
write.table(T2aResMOBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_2a_MO.bed", sep = "\t", row.names = FALSE)


T2xResFCBed <- t(data.frame(strsplit(rownames(T2xResF[(T2xResF$log2FoldChange < 0) & (T2xResF$padj < 0.1),]), ":")))
rownames(T2xResFCBed) <- NULL
nrow(T2xResFCBed)
write.table(T2xResFCBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_2x_FC.bed", sep = "\t", row.names = FALSE)
T2xResFOBed <- t(data.frame(strsplit(rownames(T2xResF[(T2xResF$log2FoldChange > 0) & (T2xResF$padj < 0.1),]), ":")))
rownames(T2xResFOBed) <- NULL
nrow(T2xResFOBed)
write.table(T2xResFOBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_2x_FO.bed", sep = "\t", row.names = FALSE)

T2xResMCBed <- t(data.frame(strsplit(rownames(T2xResM[(T2xResM$log2FoldChange < 0) & (T2xResM$padj < 0.1),]), ":")))
rownames(T2xResMCBed) <- NULL
nrow(T2xResMCBed)
write.table(T2xResMCBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_2x_MC.bed", sep = "\t", row.names = FALSE)
T2xResMOBed <- t(data.frame(strsplit(rownames(T2xResM[(T2xResM$log2FoldChange > 0) & (T2xResM$padj < 0.1),]), ":")))
rownames(T2xResMOBed) <- NULL
nrow(T2xResMOBed)
write.table(T2xResMOBed, "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/AgeATAC/BedRes/Type_2x_MO.bed", sep = "\t", row.names = FALSE)

[1] 527

[1] 101

[1] 105

[1] 52

[1] 545

...

[1] 565

[1] 4

[1] 7

[1] 528

[1] 202

In [63]:
head(T2xResM)
nrow(T2xResM[(T2xResM$log2MoldChange < 0) & (T2xResM$padj < 0.1),])

log2 fold change (MMSE): age 
Wald test p-value: age 
DataFrame with 6 rows and 5 columns
                           baseMean log2FoldChange      lfcSE    pvalue
                          <numeric>      <numeric>  <numeric> <numeric>
chr10:100009814:100010115  67.01739   -0.000484541 0.00156435  0.140324
chr10:100022425:100022726   3.64446   -0.000196039 0.00202533  0.653521
chr10:100025760:100026061   4.42817    0.000774535 0.00287948  0.187141
chr10:100046058:100046359  23.46847   -0.000133915 0.00104113  0.567340
chr10:100115078:100115379   3.05086   -0.001998668 0.00461090  0.030108
chr10:100127715:100128016   4.25110   -0.000592450 0.00261825  0.266100
                               padj
                          <numeric>
chr10:100009814:100010115  0.674601
chr10:100022425:100022726  1.000000
chr10:100025760:100026061  1.000000
chr10:100046058:100046359  0.915818
chr10:100115078:100115379  1.000000
chr10:100127715:100128016  1.000000

[1] 0

In [22]:
ResFrameF <- rbind(T1ResF, T2aResF)
ResFrameF <- rbind(ResFrameF, T2xResF)
ResFrameM <- rbind(T1ResM, T2aResM)
ResFrameM <- rbind(ResFrameM, T2xResM)

ResFrameTot <- rbind(ResFrameF, ResFrameM)

OpenFrame <- ResFrameTot[(ResFrameTot$log2FoldChange < 0) & (ResFrameTot$padj < 0.1),]
CloseFrame <- ResFrameTot[(ResFrameTot$log2FoldChange > 0) & (ResFrameTot$padj < 0.1),]

PlotFrame <- data.frame(Pattern = c("Close", "Open"), Count = c(nrow(OpenFrame), nrow(CloseFrame)))
head(PlotFrame)

,Pattern,Count
,<chr>,<int>
1,Close,3113
2,Open,1139


In [24]:
ColorCode_FemaleUP <- "#CC6677"
ColorCode_FemaleDOWN <- "#4AE4CB"
ColorCode_MaleUP <- "#882255"
ColorCode_MaleDOWN <- "#117733"

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Plots/AgeATAC/DESeq2PeakCompare_Total.pdf"), width=5, height=5)
mygg <- ggplot(PlotFrame, aes(x = Pattern, y = Count, fill = Pattern)) +
  geom_bar(position = 'dodge', stat = "identity") +
  xlab("Chromatin Behavior") + ylab("Number of Peaks") + ggtitle(glue("Significant FDR < 0.1 {feature} Peaks")) + 
  scale_fill_manual(values = c(ColorCode_FemaleUP,ColorCode_FemaleDOWN,ColorCode_MaleUP, ColorCode_MaleDOWN)) + 
  scale_y_continuous(breaks=seq(0,5000,500), limits = c(0,5000)) + theme_bw()
mygg
dev.off()

pdf 
  2

In [25]:
batchCompDat <- merge(data.frame(T1ResF), data.frame(T1ResM), by= "row.names") %>% dplyr::select(pvalue.x, log2FoldChange.x, padj.x, pvalue.y, log2FoldChange.y, padj.y)
batchCompDat$modp.x <- -log10(batchCompDat$pvalue.x)*sign(batchCompDat$log2FoldChange.x)
batchCompDat$modp.y <- -log10(batchCompDat$pvalue.y)*sign(batchCompDat$log2FoldChange.y)
head(batchCompDat)
if (nrow(batchCompDat[(is.na(batchCompDat$padj.x)),]) > 0) {
    batchCompDat[(is.na(batchCompDat$padj.x)),]$padj.x <- 1
}
if (nrow(batchCompDat[(is.na(batchCompDat$padj.y)),]) > 0) {
    batchCompDat[(is.na(batchCompDat$padj.y)),]$padj.y <- 1
}
print("One")
batchCompDat$ColorBlock <- rep("Niether", nrow(batchCompDat))
batchCompDat[batchCompDat$padj.x < 0.1,]$ColorBlock <- "FDR < 0.1 in Males"
batchCompDat[batchCompDat$padj.y < 0.1,]$ColorBlock <- "FDR < 0.1 in Females"
batchCompDat[(batchCompDat$padj.y < 0.1) & (batchCompDat$padj.x < 0.1),]$ColorBlock <- "FDR < 0.1 in Both"
batchCompDatPrint <- batchCompDat %>% arrange(pvalue.y)
batchCompDat <- batchCompDat[batchCompDat$ColorBlock != "Niether",]
ourLine <- lm(batchCompDat$modp.y~batchCompDat$modp.x-1)
summary(ourLine)$coefficients[1]
pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Plots/AgeATAC/MvFPvalCompare_Type1.pdf"), width=6, height=6)
    mygg <- ggplot(data=batchCompDat, aes(x=-log10(pvalue.x)*sign(log2FoldChange.x), y=-log10(pvalue.y)*sign(log2FoldChange.y))) +
        geom_point(aes(color = ColorBlock), alpha = 0.3)+#, position = position_jitter(width = 0.01)) +
        geom_abline(intercept = 0, slope = 1, alpha = 1, col = "black") +
        geom_abline(intercept = 0, slope = summary(ourLine)$coefficients[1], alpha = 1, col = "red")+
        theme_bw() + xlim(-20,20) + ylim(-20,20) +
        xlab("Female -log10(Nominal P-value)*sign(log2FC)") + ylab("Male -log10(Nominal P-value)*sign(log2FC)") + ggtitle(glue("Comparing Nominal P-values Between Sexes")) 
   print(mygg)
dev.off()


,pvalue.x,log2FoldChange.x,padj.x,pvalue.y,log2FoldChange.y,padj.y,modp.x,modp.y
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0.30933418,4.340395e-04,0.7877627,0.2166882,2.378281e-04,0.8420990,0.50957210,0.6641646
2,0.36967888,4.913343e-04,1.0000000,0.2592086,2.420761e-04,0.8626969,0.43217536,0.5863507
3,0.01653026,-1.768400e-03,0.3321643,0.0525336,3.805148e-04,0.6558560,-1.78172029,1.2795628
4,0.81742408,-2.823698e-05,0.9675069,0.5379274,-7.314425e-05,0.9444701,-0.08755257,-0.2692763
5,0.17530022,8.567080e-04,1.0000000,0.7161104,-6.308463e-05,0.9726404,0.75621754,-0.1450200
6,0.82834461,9.930824e-05,0.9697753,0.4720229,1.299714e-04,0.9284705,0.08178895,0.3260369


[1] "One"


[1] 0.2079488

pdf 
  2